# Business Intelligence — Data Preparation & Concepts Tutorial

**Module:** 3.3 — BI with Tableau  
**Primary outcome:** Prepare analysis-ready datasets for BI tools like Tableau, and understand key BI concepts (dimensions, measures, aggregations, dashboards) through Python equivalents.  
**Estimated time:** 45–75 minutes

---

## What You'll Learn

- The difference between **dimensions** (categorical) and **measures** (numeric) in BI
- How to **clean and prepare data** for Tableau and other BI tools
- Why **long (tidy) format** is preferred over wide format
- How to replicate **Tableau aggregations** using pandas
- How to design and compute meaningful **KPIs**
- How to build a **Python dashboard** using Plotly subplots

---

## Datasets Used

| Dataset | Source | Description |
|---------|--------|-------------|
| `tips` | `seaborn.load_dataset('tips')` | Restaurant bill and tip records |
| `gapminder` | `plotly.express.data.gapminder()` | Global country development metrics 1952–2007 |


## Section 1 — What is Business Intelligence?

### BI Tools vs Python

**Business Intelligence (BI)** is the practice of collecting, transforming, and visualising data to support decision-making. Common BI platforms include:

| Tool | Strengths | Typical User |
|------|-----------|-------------|
| **Tableau** | Beautiful interactive dashboards, drag-and-drop, fast | Analysts, managers |
| **Power BI** | Deep Microsoft integration, cost-effective | Corporate / Office 365 shops |
| **Looker / Looker Studio** | Web-based, strong Google Cloud integration | Data teams using GCP |
| **Python (pandas + Plotly)** | Maximum flexibility, automation, reproducibility | Data scientists / engineers |

### When to use each?

- **BI tools** — when non-technical stakeholders need self-service dashboards, quick exploration, or scheduled reports without code.
- **Python** — when you need complex transformations, machine learning integration, automation pipelines, or version-controlled reproducible analysis.

In practice, **Python and BI tools complement each other**: Python handles data cleaning and feature engineering; Tableau handles interactive presentation.

---

### Key Vocabulary

| Term | Definition | Example |
|------|------------|--------|
| **Dimension** | A categorical field used to slice/group data | `day`, `gender`, `country` |
| **Measure** | A numeric field that can be aggregated | `total_bill`, `tip`, `population` |
| **Aggregation** | A function that collapses many rows into one value | SUM, AVG, COUNT, MAX |
| **Granularity** | The level of detail in a dataset | Per transaction vs per day vs per month |
| **Dashboard** | A single view combining multiple charts/KPIs | Executive sales dashboard |
| **KPI** | Key Performance Indicator — a metric tied to a business goal | Average tip rate |

> **Tableau Rule of Thumb:** Drag a **dimension** to Rows/Columns to group. Drag a **measure** to the chart area to aggregate.


## Section 2 — Dataset Overview

We load both datasets, perform quick EDA, and identify dimensions vs measures.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# Load datasets
tips = sns.load_dataset('tips')
gapminder = px.data.gapminder()

print('tips shape:', tips.shape)
print('gapminder shape:', gapminder.shape)

In [ ]:
# Preview tips dataset
print('=== Tips Dataset (first 5 rows) ===')
tips.head()

In [ ]:
# Preview gapminder dataset
print('=== Gapminder Dataset (first 5 rows) ===')
gapminder.head()

In [ ]:
# Basic info and dtypes for tips
print('=== Tips Dtypes ===')
print(tips.dtypes)
print()
print('=== Tips Describe ===')
tips.describe()

In [ ]:
# Basic info for gapminder
print('=== Gapminder Dtypes ===')
print(gapminder.dtypes)
print()
print('=== Gapminder Describe ===')
gapminder.describe()

### Identifying Dimensions vs Measures

A quick way to do this programmatically:

In [ ]:
def classify_columns(df, name):
    """Print dimension/measure classification for a dataframe."""
    dimensions = df.select_dtypes(include=['object', 'category']).columns.tolist()
    measures = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f'=== {name} ===')
    print(f'Dimensions (categorical): {dimensions}')
    print(f'Measures   (numeric):     {measures}')
    print()

classify_columns(tips, 'Tips Dataset')
classify_columns(gapminder, 'Gapminder Dataset')

> **Note:** Some numeric columns are actually dimensions in disguise. For example, `year` in gapminder is numeric but acts as a time dimension. `size` in tips is numeric but behaves as an ordinal category. Always apply domain knowledge alongside dtype checks.

## Section 3 — Data Preparation for BI

Before loading data into Tableau (or any BI tool), follow these preparation steps:

1. **Fix dtypes** — ensure categoricals are strings/categories, numerics are `float`/`int`
2. **Rename columns** — Tableau-friendly names: no spaces, consistent case
3. **Remove nulls or document them** — BI tools handle nulls poorly in aggregations
4. **Create derived columns** — compute any fields Tableau would calculate on the fly
5. **Export to CSV** — the universal BI input format

In [ ]:
# Step 1: Copy the raw dataset — never mutate originals
tips_clean = tips.copy()

# Check for nulls
print('Null values in tips:')
print(tips_clean.isnull().sum())

In [ ]:
# Step 2: Fix dtypes
# Convert category columns to plain string for CSV compatibility
cat_cols = tips_clean.select_dtypes('category').columns
tips_clean[cat_cols] = tips_clean[cat_cols].astype(str)

# Treat 'size' as integer (it already is, but be explicit)
tips_clean['size'] = tips_clean['size'].astype(int)

print('Dtypes after fixing:')
print(tips_clean.dtypes)

In [ ]:
# Step 3: Rename columns to Tableau-friendly format
# Convention: snake_case, no spaces, lowercase
tips_clean.rename(columns={
    'total_bill': 'total_bill',   # already good
    'tip':        'tip',
    'sex':        'gender',       # rename 'sex' to 'gender' — more readable in dashboards
    'smoker':     'is_smoker',
    'day':        'day_of_week',
    'time':       'meal_time',
    'size':       'party_size'
}, inplace=True)

print('Renamed columns:', tips_clean.columns.tolist())

In [ ]:
# Step 4: Create derived columns
# These are common Tableau 'calculated fields' — we pre-compute them for speed

# Tip percentage — classic KPI for restaurant data
tips_clean['tip_pct'] = (tips_clean['tip'] / tips_clean['total_bill'] * 100).round(2)

# Bill per person — normalises for party size
tips_clean['bill_per_person'] = (tips_clean['total_bill'] / tips_clean['party_size']).round(2)

# Tip per person
tips_clean['tip_per_person'] = (tips_clean['tip'] / tips_clean['party_size']).round(2)

# Is it a high tip? (> 20% = generous)
tips_clean['is_high_tip'] = tips_clean['tip_pct'] > 20

print('New columns added:')
tips_clean[['total_bill', 'tip', 'party_size', 'tip_pct', 'bill_per_person', 'tip_per_person', 'is_high_tip']].head(8)

In [ ]:
# Step 5: Export to CSV
tips_clean.to_csv('tips_clean.csv', index=False)
print('Exported: tips_clean.csv')
print(f'Shape: {tips_clean.shape[0]} rows x {tips_clean.shape[1]} columns')
tips_clean.head()

> **Tableau Tip:** When you connect to `tips_clean.csv`, Tableau will auto-detect numeric columns as measures and string columns as dimensions. The derived columns we computed (`tip_pct`, `bill_per_person`) are ready to drag straight onto the canvas.

## Section 4 — Wide vs Long Format

### Why does format matter?

BI tools like Tableau are designed around **tidy / long format** data:
- Each row = one observation
- Each column = one variable

**Wide format** spreads a variable's values across multiple columns — useful for spreadsheets but awkward for BI tools.

| Format | Structure | Best for |
|--------|-----------|----------|
| **Wide** | One column per year/category | Excel pivot tables, quick scanning |
| **Long (tidy)** | One row per observation | Tableau, pandas, ggplot, SQL |

Let's demonstrate with the gapminder dataset.

In [ ]:
# Current gapminder format — already LONG (tidy)
print('=== Long format (original gapminder) ===')
print('Each row = one country in one year')
gapminder[['country', 'year', 'lifeExp', 'gdpPercap', 'pop']].head(12)

In [ ]:
# Convert to WIDE format: pivot so each year becomes a column
gap_wide = gapminder.pivot_table(
    index='country',
    columns='year',
    values='lifeExp'
).reset_index()

gap_wide.columns.name = None   # clean up the column index name

print('=== Wide format (lifeExp per year) ===')
print('Each column = one year, each row = one country')
gap_wide.head(6)

In [ ]:
# Convert WIDE back to LONG using pd.melt()
gap_long = pd.melt(
    gap_wide,
    id_vars=['country'],       # columns to keep as-is
    var_name='year',           # name for the new 'variable' column
    value_name='lifeExp'       # name for the new 'value' column
)

gap_long['year'] = gap_long['year'].astype(int)

print('=== Back to long format via pd.melt() ===')
print(f'Shape: {gap_long.shape}')
gap_long.sort_values(['country', 'year']).head(12)

### pd.melt() — The Key Function

```python
pd.melt(
    df,
    id_vars=[...],     # columns that stay as columns (identifiers)
    var_name='...',    # name for the column that will hold the old column headers
    value_name='...'   # name for the column that will hold the values
)
```

> **Tableau Rule:** If your data is in wide format and your chart isn't working as expected, `pd.melt()` it first, re-export to CSV, and reconnect Tableau.


## Section 5 — Aggregations: Tableau's Core

In Tableau, when you drag a measure onto the canvas, it **automatically aggregates** (SUM by default). Understanding aggregations in Python lets you validate what Tableau is computing.

We'll replicate four common Tableau patterns:
1. Basic `groupby` + `agg`
2. Window functions with `transform`
3. Running totals
4. Percentage of total

In [ ]:
# Pattern 1: Basic groupby + agg
# Tableau equivalent: drag 'day_of_week' to Rows, drag 'total_bill' to Columns (SUM)

daily_summary = tips_clean.groupby('day_of_week').agg(
    total_revenue   = ('total_bill',  'sum'),
    avg_bill        = ('total_bill',  'mean'),
    total_tips      = ('tip',         'sum'),
    avg_tip_pct     = ('tip_pct',     'mean'),
    transaction_cnt = ('total_bill',  'count')
).round(2).reset_index()

print('=== Daily Revenue Summary (Tableau: groupby Day) ===')
daily_summary

In [ ]:
# Pattern 2: Window function with transform
# Tableau equivalent: WINDOW_AVG() or Level of Detail (LOD) expressions
# Goal: add a 'group average' column WITHOUT collapsing rows

tips_clean['avg_bill_by_day'] = tips_clean.groupby('day_of_week')['total_bill'].transform('mean').round(2)
tips_clean['vs_day_avg'] = (tips_clean['total_bill'] - tips_clean['avg_bill_by_day']).round(2)

print('=== Window Function: each row vs its day-group average ===')
tips_clean[['day_of_week', 'total_bill', 'avg_bill_by_day', 'vs_day_avg']].head(10)

In [ ]:
# Pattern 3: Running total
# Tableau equivalent: RUNNING_SUM(SUM([total_bill]))

daily_ordered = daily_summary.copy()

# Define a logical day order
day_order = ['Thur', 'Fri', 'Sat', 'Sun']
daily_ordered['day_of_week'] = pd.Categorical(daily_ordered['day_of_week'], categories=day_order, ordered=True)
daily_ordered = daily_ordered.sort_values('day_of_week')

daily_ordered['running_revenue'] = daily_ordered['total_revenue'].cumsum().round(2)

print('=== Running Total (Tableau: RUNNING_SUM) ===')
daily_ordered[['day_of_week', 'total_revenue', 'running_revenue']]

In [ ]:
# Pattern 4: Percentage of total
# Tableau equivalent: SUM([total_bill]) / TOTAL(SUM([total_bill]))

grand_total = daily_ordered['total_revenue'].sum()
daily_ordered['pct_of_total'] = (daily_ordered['total_revenue'] / grand_total * 100).round(1)

print(f'=== % of Total Revenue (Grand Total: ${grand_total:.2f}) ===')
daily_ordered[['day_of_week', 'total_revenue', 'pct_of_total']]

### Summary: Python ↔ Tableau Aggregation Map

| Python | Tableau |
|--------|--------|
| `groupby().agg('sum')` | SUM(measure) dragged to canvas |
| `groupby().agg('mean')` | AVG(measure) |
| `groupby().transform('mean')` | `WINDOW_AVG()` or Fixed LOD `{FIXED [dim]: AVG([measure])}` |
| `cumsum()` | `RUNNING_SUM(SUM([measure]))` |
| `value / value.sum()` | `SUM([measure]) / TOTAL(SUM([measure]))` |


## Section 6 — KPI Design

### What Makes a Good KPI?

A KPI (Key Performance Indicator) is a **measurable value** tied to a specific business question. Good KPIs are:

- **Specific** — answers one clear question
- **Actionable** — tells you what to do differently
- **Comparable** — has a benchmark or trend
- **Timely** — refreshed frequently enough to act on

**Bad KPI:** "We made money today"  
**Good KPI:** "Average tip rate: 16.1% (benchmark: 18%) — down 2pp vs last week"

---

Let's compute 5 KPIs from the restaurant data and print them as a formatted report.

In [ ]:
# KPI 1: Average check size
avg_check = tips_clean['total_bill'].mean()

# KPI 2: Overall tip rate
overall_tip_rate = (tips_clean['tip'].sum() / tips_clean['total_bill'].sum() * 100)

# KPI 3: Peak revenue day
peak_day = daily_ordered.loc[daily_ordered['total_revenue'].idxmax(), 'day_of_week']
peak_day_revenue = daily_ordered['total_revenue'].max()

# KPI 4: Gender split (% of transactions)
gender_split = tips_clean['gender'].value_counts(normalize=True) * 100

# KPI 5: Most common party size
most_common_size = tips_clean['party_size'].mode()[0]
size_pct = (tips_clean['party_size'] == most_common_size).mean() * 100

# Print formatted report
print('=' * 50)
print('   RESTAURANT PERFORMANCE REPORT')
print('=' * 50)
print(f'  KPI 1 — Avg Check Size:      ${avg_check:.2f}')
print(f'  KPI 2 — Overall Tip Rate:    {overall_tip_rate:.1f}%')
print(f'  KPI 3 — Peak Day:            {peak_day} (${peak_day_revenue:.2f} revenue)')
print(f'  KPI 4 — Gender Split:        Male {gender_split.get("Male", 0):.1f}% / Female {gender_split.get("Female", 0):.1f}%')
print(f'  KPI 5 — Most Common Party:   {most_common_size} people ({size_pct:.1f}% of visits)')
print('=' * 50)

In [ ]:
# Bonus: KPI breakdown by dimension — the kind of table you'd build in Tableau
kpi_by_day = tips_clean.groupby('day_of_week').agg(
    transactions = ('total_bill', 'count'),
    avg_check    = ('total_bill', 'mean'),
    tip_rate_pct = ('tip_pct',   'mean')
).round(2)

kpi_by_day['tip_rate_pct'] = kpi_by_day['tip_rate_pct'].map('{:.1f}%'.format)
kpi_by_day['avg_check']    = kpi_by_day['avg_check'].map('${:.2f}'.format)

print('=== KPI Breakdown by Day ===')
kpi_by_day

## Section 7 — Python Dashboard (Plotly Subplots)

Tableau's core product is the **dashboard** — multiple charts in a single view. Here we replicate that with Plotly's `make_subplots`.

Our 4-panel dashboard covers the restaurant dataset:
- **Panel 1** (top-left): Total bills by day — a bar chart
- **Panel 2** (top-right): Total bill vs tip coloured by meal time — a scatter
- **Panel 3** (bottom-left): Tip % distribution by day — box plots
- **Panel 4** (bottom-right): Smoker split — a pie chart

In [ ]:
# Prepare aggregated data for Panel 1
day_order = ['Thur', 'Fri', 'Sat', 'Sun']
bills_by_day = (
    tips_clean
    .groupby('day_of_week', observed=True)['total_bill']
    .sum()
    .reindex(day_order)
    .reset_index()
)

# Prepare smoker data for Panel 4
smoker_counts = tips_clean['is_smoker'].value_counts().reset_index()
smoker_counts.columns = ['is_smoker', 'count']
smoker_counts['label'] = smoker_counts['is_smoker'].map({'Yes': 'Smoker', 'No': 'Non-smoker'})

# Build subplots layout
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Total Bills by Day',
        'Total Bill vs Tip (by Meal Time)',
        'Tip % Distribution by Day',
        'Smoker Split'
    ],
    specs=[
        [{'type': 'bar'},      {'type': 'scatter'}],
        [{'type': 'box'},      {'type': 'pie'}]
    ]
)

# --- Panel 1: Bar — total bills by day ---
fig.add_trace(
    go.Bar(
        x=bills_by_day['day_of_week'],
        y=bills_by_day['total_bill'],
        marker_color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA'],
        name='Total Bills',
        showlegend=False
    ),
    row=1, col=1
)

# --- Panel 2: Scatter — total bill vs tip, coloured by meal_time ---
for meal_time, color in [('Dinner', '#EF553B'), ('Lunch', '#636EFA')]:
    subset = tips_clean[tips_clean['meal_time'] == meal_time]
    fig.add_trace(
        go.Scatter(
            x=subset['total_bill'],
            y=subset['tip'],
            mode='markers',
            marker=dict(color=color, size=6, opacity=0.7),
            name=meal_time
        ),
        row=1, col=2
    )

# --- Panel 3: Box — tip % by day ---
for day in day_order:
    subset = tips_clean[tips_clean['day_of_week'] == day]
    fig.add_trace(
        go.Box(
            y=subset['tip_pct'],
            name=day,
            showlegend=False
        ),
        row=2, col=1
    )

# --- Panel 4: Pie — smoker split ---
fig.add_trace(
    go.Pie(
        labels=smoker_counts['label'],
        values=smoker_counts['count'],
        hole=0.3,
        marker_colors=['#EF553B', '#636EFA'],
        showlegend=True
    ),
    row=2, col=2
)

# Layout polish
fig.update_layout(
    title_text='Restaurant Performance Dashboard',
    title_font_size=18,
    height=650,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5)
)

fig.update_xaxes(title_text='Day', row=1, col=1)
fig.update_yaxes(title_text='Total Bills ($)', row=1, col=1)
fig.update_xaxes(title_text='Total Bill ($)', row=1, col=2)
fig.update_yaxes(title_text='Tip ($)', row=1, col=2)
fig.update_yaxes(title_text='Tip %', row=2, col=1)

fig.show()

> **Design Notes:**
> - The bar chart answers: **Which day generates the most revenue?**
> - The scatter answers: **Is there a linear relationship between bill and tip? Does it differ by meal time?**
> - The box plots answer: **Is tip generosity consistent across days, or are there outliers?**
> - The pie answers: **What share of customers smoke?** (affects seating, capacity planning)
>
> In Tableau you'd build this by adding sheets to a Dashboard container. The interactivity (filters, tooltips, cross-highlighting) is where Tableau shines over static Python charts.

## Section 8 — Preparing Gapminder for Tableau

### Goal

Prepare a clean, Tableau-ready gapminder export focused on 5 countries. We'll also add ISO country codes so Tableau's built-in **Map** chart type works automatically.

### Countries Selected

Australia, Singapore, Japan, United Kingdom, United States — a mix of regions with different development trajectories.

### What Charts Would You Build in Tableau?

With this dataset you'd typically build:

1. **Line Chart** — `year` on X, `lifeExp` on Y, one line per `country`. Drag `country` to Color. Shows life expectancy trends over time.
2. **Scatter Plot (Bubble Chart)** — `gdpPercap` on X, `lifeExp` on Y, `pop` as Size, `country` as Color. Tableau's built-in version of Hans Rosling's famous chart.
3. **Map** — Drag `iso_alpha` (or country name) to the canvas; Tableau auto-geocodes. Colour by `lifeExp` in the most recent year.
4. **Bar Chart** — `country` on Rows, `gdpPercap` on Columns, filtered to one year. Shows GDP per capita ranking.
5. **Dual-Axis Chart** — `year` on X, `lifeExp` on left Y, `gdpPercap` on right Y. Shows correlation between wealth and health.

In [ ]:
# Filter to 5 countries of interest
countries_of_interest = ['Australia', 'Singapore', 'Japan', 'United Kingdom', 'United States']
gap_filtered = gapminder[gapminder['country'].isin(countries_of_interest)].copy()

print(f'Filtered shape: {gap_filtered.shape}')
print('Years covered:', sorted(gap_filtered['year'].unique()))

In [ ]:
# Add ISO alpha-3 codes for Tableau's map chart
iso_map = {
    'Australia':      'AUS',
    'Singapore':      'SGP',
    'Japan':          'JPN',
    'United Kingdom': 'GBR',
    'United States':  'USA'
}
gap_filtered['iso_alpha'] = gap_filtered['country'].map(iso_map)

# Rename columns for Tableau clarity
gap_filtered.rename(columns={
    'lifeExp':   'life_expectancy',
    'gdpPercap': 'gdp_per_capita',
    'pop':       'population'
}, inplace=True)

# Add a derived column: GDP rank within each year
gap_filtered['gdp_rank_in_year'] = (
    gap_filtered.groupby('year')['gdp_per_capita']
    .rank(ascending=False, method='dense')
    .astype(int)
)

gap_filtered.head(10)

In [ ]:
# Export to CSV
gap_filtered.to_csv('gapminder_tableau.csv', index=False)
print('Exported: gapminder_tableau.csv')
print(f'Shape: {gap_filtered.shape[0]} rows x {gap_filtered.shape[1]} columns')
print('\nColumns:', gap_filtered.columns.tolist())

In [ ]:
# Quick Python preview of the charts we'd build in Tableau

# Chart A: Life expectancy trends
fig_trends = px.line(
    gap_filtered,
    x='year', y='life_expectancy',
    color='country',
    title='Life Expectancy Trends (1952–2007)',
    labels={'life_expectancy': 'Life Expectancy (years)', 'year': 'Year'},
    template='plotly_white'
)
fig_trends.show()

In [ ]:
# Chart B: Bubble chart (Hans Rosling style)
fig_bubble = px.scatter(
    gap_filtered[gap_filtered['year'] == 2007],
    x='gdp_per_capita', y='life_expectancy',
    size='population', color='country',
    text='iso_alpha',
    title='GDP per Capita vs Life Expectancy (2007)',
    labels={
        'gdp_per_capita': 'GDP per Capita (USD)',
        'life_expectancy': 'Life Expectancy (years)'
    },
    template='plotly_white',
    size_max=60
)
fig_bubble.update_traces(textposition='top center')
fig_bubble.show()

### Tableau Steps (for reference)

To build the bubble chart in Tableau after connecting to `gapminder_tableau.csv`:

1. Connect > Text File > select `gapminder_tableau.csv`
2. Drag `gdp_per_capita` → Columns
3. Drag `life_expectancy` → Rows
4. Drag `population` → Size (Marks card)
5. Drag `country` → Color (Marks card)
6. Drag `year` → Filters shelf → select 2007
7. Change mark type to **Circle**

The result is Hans Rosling's Gapminder chart — in under 2 minutes.

## Section 9 — Practice Exercises

Work through these exercises on your own. Solutions are provided below each exercise — try before looking!

---

### Exercise 1 — Data Preparation

Using the raw `tips` dataset:

1. Create a new column `meal_revenue_rank` that ranks each transaction by `total_bill` within its `meal_time` group (Dinner vs Lunch), with rank 1 = highest bill.
2. Create a boolean column `is_weekend` that is `True` when `day` is `Sat` or `Sun`.
3. Export the result to `tips_exercise1.csv`.

**Hint:** Use `groupby().rank()` and a list-membership check with `.isin()`.

In [ ]:
# Your code here


In [ ]:
# --- SOLUTION Exercise 1 ---

ex1 = tips.copy()

# 1. Rank within meal_time group by total_bill (descending = rank 1 = highest)
ex1['meal_revenue_rank'] = (
    ex1.groupby('time')['total_bill']
    .rank(ascending=False, method='dense')
    .astype(int)
)

# 2. Weekend flag
ex1['is_weekend'] = ex1['day'].isin(['Sat', 'Sun'])

# 3. Export
ex1.to_csv('tips_exercise1.csv', index=False)

print('Exercise 1 solution:')
ex1[['total_bill', 'time', 'day', 'meal_revenue_rank', 'is_weekend']].head(10)

---

### Exercise 2 — Wide vs Long Format

Using the full `gapminder` dataset:

1. Create a **wide** pivot table where rows are countries, columns are years, and values are `gdpPercap`. Keep only years 1987, 1997, 2007.
2. Melt it back to long format.
3. Add a column `decade` to the long format result (e.g. 1987 → "1980s", 1997 → "1990s", 2007 → "2000s").

**Hint:** `pd.pivot_table()`, `pd.melt()`, and a dictionary map or `//10*10` arithmetic.

In [ ]:
# Your code here


In [ ]:
# --- SOLUTION Exercise 2 ---

# 1. Wide pivot for 3 years
gap_3yr = gapminder[gapminder['year'].isin([1987, 1997, 2007])]

wide_gdp = gap_3yr.pivot_table(
    index='country',
    columns='year',
    values='gdpPercap'
).reset_index()
wide_gdp.columns.name = None

print('Wide format (first 5 rows):')
print(wide_gdp.head())

# 2. Melt back to long
long_gdp = pd.melt(
    wide_gdp,
    id_vars=['country'],
    var_name='year',
    value_name='gdpPercap'
)
long_gdp['year'] = long_gdp['year'].astype(int)

# 3. Add decade column
decade_map = {1987: '1980s', 1997: '1990s', 2007: '2000s'}
long_gdp['decade'] = long_gdp['year'].map(decade_map)

print('\nLong format with decade (first 8 rows):')
long_gdp.sort_values(['country', 'year']).head(8)

---

### Exercise 3 — Aggregations & Dashboard

Using `tips_clean`:

1. Compute the **average tip percentage** grouped by both `gender` and `meal_time` (a 2-level groupby).
2. Add a column to the result called `above_avg` that flags rows where the group's avg tip pct is above the **overall** average tip pct.
3. Build a Plotly bar chart showing these group averages with different colours for `gender` and faceted by `meal_time`.

**Hint:** `px.bar(..., facet_col='meal_time', color='gender', barmode='group')`

In [ ]:
# Your code here


In [ ]:
# --- SOLUTION Exercise 3 ---

# 1. Two-level groupby
group_tip = (
    tips_clean
    .groupby(['gender', 'meal_time'])['tip_pct']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'tip_pct': 'avg_tip_pct'})
)

# 2. Flag above-overall-average groups
overall_avg = tips_clean['tip_pct'].mean()
group_tip['above_avg'] = group_tip['avg_tip_pct'] > overall_avg

print(f'Overall avg tip pct: {overall_avg:.2f}%')
print(group_tip)

# 3. Plotly bar chart
fig_ex3 = px.bar(
    group_tip,
    x='gender', y='avg_tip_pct',
    color='gender',
    facet_col='meal_time',
    barmode='group',
    title='Average Tip % by Gender and Meal Time',
    labels={'avg_tip_pct': 'Avg Tip %', 'gender': 'Gender'},
    template='plotly_white',
    text='avg_tip_pct'
)

# Add a horizontal reference line for the overall average
fig_ex3.add_hline(
    y=overall_avg,
    line_dash='dash', line_color='grey',
    annotation_text=f'Overall avg: {overall_avg:.1f}%',
    annotation_position='bottom right'
)

fig_ex3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_ex3.show()

---

## Recap

| Concept | What you did |
|---------|-------------|
| Dimensions vs Measures | Classified columns in `tips` and `gapminder` programmatically |
| Data prep for BI | Fixed dtypes, renamed columns, created derived fields, exported CSVs |
| Wide vs Long format | Pivoted gapminder wide and melted it back to long |
| Tableau aggregations | Replicated groupby, window functions, running totals, % of total |
| KPI design | Computed 5 restaurant KPIs and printed a formatted report |
| Python dashboard | Built a 4-panel Plotly dashboard with `make_subplots` |
| Gapminder for Tableau | Filtered, enriched with ISO codes, exported |

### Next Steps

- Load `tips_clean.csv` and `gapminder_tableau.csv` into **Tableau Public** (free) and recreate the charts described in Section 8.
- Explore Tableau's **Level of Detail (LOD)** expressions — they map directly to the `groupby().transform()` pattern you practised here.
- Try connecting **multiple CSVs** with a Tableau data relationship (equivalent to a pandas `merge`).
